# RNN tagging

Run the cells in order.

Use the course-provided CoNLL files in `data/conll2003/`; see the README for setup.

## Sentences and tags

We will use an RNN to combine each word’s embedding with information from earlier words.

We use the CoNLL workshop’s 2,000 training and 500 validation sentences, sampled with seed 17.

In [ ]:
from pathlib import Path
import random, time
import torch
from torch import nn
from torch.nn.utils.rnn import pad_sequence
from seqeval.metrics import precision_score, recall_score, f1_score
from seqeval.scheme import IOB2

torch.set_num_threads(1)
SEED = 17
labels = "O B-PER I-PER B-ORG I-ORG B-LOC I-LOC B-MISC I-MISC".split()
DATA = Path("data/conll2003")
def read_conll(path):
    examples, words, tags = [], [], []
    for line in path.read_text().splitlines() + [""]:
        if not line.strip() or line.startswith("-DOCSTART-"):
            if words:
                examples.append((words, tags)); words, tags = [], []
        else:
            fields = line.split()
            words.append(fields[0]); tags.append(fields[-1])
    return examples
train = random.Random(SEED).sample(read_conll(DATA / "train.txt"), 2000)
valid = random.Random(SEED).sample(read_conll(DATA / "valid.txt"), 500)
vocab = {"<PAD>": 0, "<UNK>": 1}
for words, tags in train:
    for word in words:
        vocab.setdefault(word.lower(), len(vocab))
def batch(examples):
    ids = [torch.tensor([vocab.get(w.lower(), 1) for w in words]) for words, _ in examples]
    targets = [torch.tensor([labels.index(tag) for tag in tags]) for _, tags in examples]
    return (pad_sequence(ids, batch_first=True, padding_value=0),
            pad_sequence(targets, batch_first=True, padding_value=-100))
train_ids, train_targets = batch(train[:32])
print("Train:", len(train), "validation:", len(valid), "vocabulary:", len(vocab))
example = next(e for e in train if "I-PER" in e[1] and len(e[0]) < 15)
print(list(zip(*example)))
print("Batch IDs:", train_ids.shape, "targets:", train_targets.shape)

## Word embeddings

An embedding lookup gives the same word the same vector in every sentence. The RNN can use the words before it to figure out whether we mean a person or a place.

Here we train 32-dimensional embeddings. To start from GloVe instead, place glove.6B.100d.txt beside the notebook; the cell loads matching rows and trains them with the model.

In [ ]:
torch.manual_seed(SEED)
glove_path = Path("glove.6B.100d.txt")
embedding_dim = 100 if glove_path.exists() else 32
weights = torch.randn(len(vocab), embedding_dim) * 0.1
weights[0].zero_()
loaded = 0
if glove_path.exists():
    with glove_path.open() as source:
        for line in source:
            word, _, values = line.partition(" ")
            if word in vocab and word not in ("<PAD>", "<UNK>"):
                weights[vocab[word]] = torch.tensor([float(v) for v in values.split()])
                loaded += 1
embedding = nn.Embedding.from_pretrained(weights.clone(), freeze=False, padding_idx=0)
probe_texts = ["They visited Washington", "President Washington spoke"]
probe = [(text.split(), ["O"] * len(text.split())) for text in probe_texts]
probe_ids, _ = batch(probe)
x = embedding(probe_ids)
assert torch.equal(x[0, 2], x[1, 1])
print("Same Washington vector:", torch.equal(x[0, 2], x[1, 1]))
print("Word vectors:", x.shape)
print("Embedding width:", embedding_dim, "GloVe rows loaded:", loaded)
print("Probe unknown words:", [w for text in probe_texts for w in text.lower().split() if w not in vocab])

## The RNN

At each position, the RNN combines the current word vector with its previous hidden state.

In [ ]:
class IndependentTagger(nn.Module):
    def __init__(self):
        super().__init__()
        self.embedding = nn.Embedding.from_pretrained(weights.clone(), freeze=False, padding_idx=0)
        self.output = nn.Linear(embedding_dim, len(labels))
    def forward(self, ids):
        return self.output(self.embedding(ids))

class RNNTagger(nn.Module):
    def __init__(self):
        super().__init__()
        self.embedding = nn.Embedding.from_pretrained(weights.clone(), freeze=False, padding_idx=0)
        self.rnn = nn.RNN(embedding_dim, 32, batch_first=True)
        self.output = nn.Linear(32, len(labels))
    def forward(self, ids):
        vectors = self.embedding(ids)
        states, final_state = self.rnn(vectors)
        return self.output(states)

torch.manual_seed(SEED)
independent = IndependentTagger()
torch.manual_seed(SEED)
recurrent = RNNTagger()
print("Vectors:", recurrent.embedding(probe_ids).shape)
print("Tag scores:", recurrent(probe_ids).shape)

## Reading direction

This RNN reads from left to right. Its prediction can depend on the current word and earlier words.

A bidirectional RNN adds a second pass from right to left, giving access to later words too.

In [ ]:
recurrent.eval()
a = torch.tensor([[vocab.get(w, 1) for w in "president washington spoke".split()]])
b = torch.tensor([[vocab.get(w, 1) for w in "president washington arrived".split()]])
with torch.no_grad():
    # Both prefixes through Washington are identical.
    assert torch.allclose(recurrent(a)[0, 1], recurrent(b)[0, 1])
print("Changing a later word preserves Washington scores in this forward RNN.")
# nn.RNN(..., bidirectional=True) would return 64 features with hidden_size=32.
# Its output layer would use nn.Linear(64, len(labels)).
# For padded bidirectional batches, use packed sequences to exclude PAD steps.

## Training both models

Both models start with the same embedding weights and train on the same CoNLL sentences, batches, loss, learning rate and number of epochs. The RNN adds trainable parameters as well as context.

In [ ]:
criterion = nn.CrossEntropyLoss(ignore_index=-100)
histories = {}
orders = []
for epoch in range(5):
    order = list(range(len(train)))
    random.Random(SEED + epoch).shuffle(order)
    orders.append(order)
for name, model in [("independent", independent), ("rnn", recurrent)]:
    optimizer = torch.optim.Adam(model.parameters(), lr=0.005)
    history = []
    for epoch, order in enumerate(orders):
        model.train()
        total, tokens = 0.0, 0
        for start in range(0, len(order), 32):
            ids, targets = batch([train[i] for i in order[start:start+32]])
            optimizer.zero_grad()
            scores = model(ids)
            loss = criterion(scores.reshape(-1, len(labels)), targets.reshape(-1))
            loss.backward()
            optimizer.step()
            count = targets.ne(-100).sum().item()
            total += loss.item() * count; tokens += count
        history.append(total / tokens)
        print(name, "epoch", epoch+1, "loss", round(history[-1], 3))
    histories[name] = history
    count = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(name, "trainable parameters:", count)

## Compare Washington

Compare the scores for “Washington” in “They visited Washington” and “President Washington spoke.”

In [ ]:
for name, model in [("independent", independent), ("rnn", recurrent)]:
    model.eval()
    with torch.no_grad():
        scores = model(probe_ids)
        washington_scores = torch.stack([scores[0, 2], scores[1, 1]])
    print("\n", name)
    for text, row in zip(probe_texts, washington_scores):
        print(text, "→", labels[row.argmax().item()])
        print("B-PER:", round(row[1].item(), 3), "B-LOC:", round(row[5].item(), 3))

## Validation and errors

Compare validation entity F1 and inspect the mistakes from both models.

An entity counts as correct when its type, start and end all match. A wrong boundary gives one missed gold entity and one incorrect predicted entity.

Precision = correct entities / predicted entities. Recall = correct entities / gold entities. F1 combines precision and recall: 2PR / (P + R).

We use strict IOB2 scoring here and remove padded positions before evaluation. We evaluate on the workshop’s 500 validation sentences. Compare several seeds before drawing conclusions about the architectures.

In [ ]:
def evaluate(model, examples):
    gold, predicted = [], []
    model.eval()
    with torch.no_grad():
        for start in range(0, len(examples), 64):
            subset = examples[start:start+64]
            ids, _ = batch(subset)
            predicted_ids = model(ids).argmax(dim=-1)
            for row, (words, tags) in zip(predicted_ids, subset):
                gold.append(tags)
                predicted.append([labels[i] for i in row[:len(words)].tolist()])
    options = dict(mode="strict", scheme=IOB2, zero_division=0)
    metrics = {
        "precision": float(precision_score(gold, predicted, **options)),
        "recall": float(recall_score(gold, predicted, **options)),
        "f1": float(f1_score(gold, predicted, **options)),
    }
    return metrics, predicted
results = {}
for name, model in [("independent", independent), ("rnn", recurrent)]:
    metrics, predictions = evaluate(model, valid)
    results[name] = {"metrics": metrics, "predicted": predictions}
    print("\n", name, {key: round(value, 3) for key, value in metrics.items()})
    errors_shown = 0
    for (words, gold), pred in zip(valid, predictions):
        if gold != pred:
            print(" ".join(words))
            print("gold:", gold)
            print("pred:", pred)
            errors_shown += 1
            if errors_shown == 3: break
# Correct type, incomplete boundary: the person entity is missed.
gold_boundary = [["B-PER", "I-PER", "O"]]
wrong_boundary = [["B-PER", "O", "O"]]
print("Boundary example F1:", f1_score(gold_boundary, wrong_boundary,
                                      mode="strict", scheme=IOB2, zero_division=0))